# 06 GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted. Notebook 06 builds SI from all screened yearly pairs and applies its own explicit fit and plot filters, independently of notebook 05 pair-selection thresholds.

## Inputs

- Screened yearly pairs and background publication counts from notebook 04.
- GLASS pair data and the configured pathogen/antibiotic lookup tables.

## Outputs

- Notebook 06 SI/rank caches, interactive GLASS comparison figures, PDF exports, and supporting CSV tables under the configured `outputs/` directory.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    build_yearly_si_dataset,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    load_yearly_si_inputs,
    prepare_antibiotic_analysis_level,
    plot_future_si_vs_si_shift,
    plot_latest_si_vs_future_si,
    summarize_pathogen_coverage,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config

from amr.screening.calibration import validate_resistance_screen_provenance


## Parameters

In [3]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
FIGURE_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.figure_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)

ANTIBIOTIC_ANALYSIS_LEVEL = cfg(PIPELINE_CONFIG, "glass_interactive.antibiotic_analysis_level")
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
SCREENING_RUN_ID = str(cfg(PIPELINE_CONFIG, "system1.run_id", ""))
SCREENING_QUESTION_VERSION = str(cfg(PIPELINE_CONFIG, "system1.question_version", ""))
SCREENING_THRESHOLD = float(cfg(PIPELINE_CONFIG, "screening.system1_probability_threshold", 0.5))
SCREENING_TARGET_PRECISION = float(cfg(PIPELINE_CONFIG, "system1.target_precision", 0.90))
SCREENING_CALIBRATION_STATUS = str(cfg(PIPELINE_CONFIG, "system1.threshold_calibration_status", "pending"))
if SCREENING_CALIBRATION_STATUS not in {"validated", "provisional_uncalibrated"}:
    raise RuntimeError("Notebook 06 requires a validated cutoff or explicit provisional_uncalibrated mode.")
if SCREENING_CALIBRATION_STATUS == "provisional_uncalibrated":
    print("PROVISIONAL EXPLORATORY OUTPUT: screening cutoff has not passed human calibration.")
validate_resistance_screen_provenance(
    YEARLY_SCREENED_PATH.with_suffix(".provenance.json"),
    run_id=SCREENING_RUN_ID,
    question_version=SCREENING_QUESTION_VERSION,
    probability_threshold=SCREENING_THRESHOLD,
    target_precision=SCREENING_TARGET_PRECISION,
    allow_provisional=SCREENING_CALIBRATION_STATUS == "provisional_uncalibrated",
)
PATHOGEN_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogen_background", PROJECT_ROOT)
ANTIBIOTIC_BG_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_background", PROJECT_ROOT)
YEARLY_SI_NB06_PATH = SI_PATHS["yearly_si"].with_name("yearly_si_nb06.parquet")
# Notebook 06 has independent caches; it does not overwrite notebook 05 outputs.
RANK_OUTPUT_PATH = SI_PATHS["rank_output"].with_name("yearly_si_ranked_future_nb06.parquet")

GLASS_XLSX_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_xlsx", PROJECT_ROOT)
GLASS_CLEANED_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_cleaned", PROJECT_ROOT)
BACTERIA_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)

# Model/forecast settings. These define the fit window and curve, not Notebook 05's
# interesting-pair selection.
FIT_START_YEAR = cfg(PIPELINE_CONFIG, "si.fit_start_year")
FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.fit_end_year")
CURVE_FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.curve_fit_end_year")
NUM_FUTURE_YEARS = cfg(PIPELINE_CONFIG, "glass_interactive.num_future_years")
LOESS_FRAC = cfg(PIPELINE_CONFIG, "si.loess_frac")
LOESS_DEGREE = cfg(PIPELINE_CONFIG, "si.loess_degree")
TAIL_DAMPING_YEARS = cfg(PIPELINE_CONFIG, "si.tail_damping_years")

# Notebook 06 pair-fit eligibility. These are local controls, independent of
# notebook 05's thresholds for selecting interesting pairs.
MIN_AMR_PAIR_PUBLICATION_COUNT = cfg(PIPELINE_CONFIG, "glass_interactive.min_amr_pair_publication_count")
MIN_NUM_YEARS_WITH_VALID_COUNTS = cfg(PIPELINE_CONFIG, "glass_interactive.min_num_years_with_valid_counts")

# Notebook 06 plot inclusion criteria, applied after fitting.
MIN_PLOT_SI = cfg(PIPELINE_CONFIG, "glass_interactive.min_plot_si")
MIN_LAST_AVAILABLE_COUNT = cfg(PIPELINE_CONFIG, "glass_interactive.min_last_available_count")
MIN_TOTAL_PUBLICATIONS = cfg(PIPELINE_CONFIG, "glass_interactive.min_total_publications")
MIN_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.min_count_abs_pct_error_mean")
MAX_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.max_count_abs_pct_error_mean")
MIN_SI_DELTA = cfg(PIPELINE_CONFIG, "glass_interactive.min_si_delta")

# Plot presentation controls (not pair eligibility filters).
PREDICTED_FUTURE_SI_THRESHOLD = cfg(PIPELINE_CONFIG, "glass_interactive.predicted_future_si_threshold")
PREDICTED_FUTURE_DELTA_SI_THRESHOLD = cfg(PIPELINE_CONFIG, "glass_interactive.predicted_future_delta_si_threshold")
SHOW_MEAN_COUNT_PCT_ERROR = cfg(PIPELINE_CONFIG, "glass_interactive.show_mean_count_pct_error")
SHOW_LABEL_BACKGROUND = cfg(PIPELINE_CONFIG, "glass_interactive.show_label_background")
MIN_COUNTS_BACKGROUND = cfg(PIPELINE_CONFIG, "glass_interactive.min_counts_background")
MIN_ABS_DELTA_SI = cfg(PIPELINE_CONFIG, "glass_interactive.min_abs_delta_si")
DISPLAY_NAME = cfg(PIPELINE_CONFIG, "glass_interactive.display_name")
MAX_TEXT_LABELS = cfg(PIPELINE_CONFIG, "glass_interactive.max_text_labels")
WIDTH = cfg(PIPELINE_CONFIG, "glass_interactive.width")
HEIGHT = cfg(PIPELINE_CONFIG, "glass_interactive.height")

for required_input_path in (YEARLY_SCREENED_PATH, PATHOGEN_BG_PATH, ANTIBIOTIC_BG_PATH):
    if not required_input_path.exists():
        raise FileNotFoundError(
            f"Missing {required_input_path}. Run notebook 04 to create screened AMR and background counts first."
        )


PROVISIONAL EXPLORATORY OUTPUT: screening cutoff has not passed human calibration.


## Load SI And Clean GLASS List

In [4]:
yearly_screened_df, pathogen_bg_df, antibiotic_bg_df, _ = load_yearly_si_inputs(
    yearly_screened_path=YEARLY_SCREENED_PATH,
    pathogen_background_path=PATHOGEN_BG_PATH,
    antibiotic_background_path=ANTIBIOTIC_BG_PATH,
)

yearly_screened_model_df, antibiotic_bg_model_df, _, analysis_summary = prepare_antibiotic_analysis_level(
    yearly_screened_df=yearly_screened_df,
    antibiotic_background_df=antibiotic_bg_df,
    unified_decisions_df=pd.DataFrame(),
    coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    analysis_level=ANTIBIOTIC_ANALYSIS_LEVEL,
)
si_df, _, _ = build_yearly_si_dataset(
    yearly_screened_model_df,
    pathogen_bg_df,
    antibiotic_bg_model_df,
    min_amr_pair_publication_count=MIN_AMR_PAIR_PUBLICATION_COUNT,
    output_path=YEARLY_SI_NB06_PATH,
)
_, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_NB06_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)

si_pathogen_summary_df = summarize_pathogen_coverage(
    source_df=yearly_screened_model_df,
    modeled_df=si_df,
    source_label="all screened AMR input pairs",
    modeled_label="Notebook 06 yearly SI input",
)
display(si_pathogen_summary_df.query("status != 'present_in_both'"))
display(glass_pairs_df.head(20))

yearly_screened rows: 21,459
yearly_screened unique pathogens: 529
yearly_screened unique antibiotics: 189
yearly_screened year range: 2000 - 2026
pathogen bg rows: 21,384
antibiotic bg rows: 6,264
unified decision rows available for driver analysis: 0
Antibiotic analysis level: individual antibiotic.
model yearly rows: 21,459
model pathogen-antibiotic rows: 5,826
model antibiotic values: 189
model antibiotic background rows: 6,264
model unified decision rows: 0


/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:229: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/linear_model/_base.py:297: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/linear_model/_base.py:297: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/linear_model/_base.py:297: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


Loaded SI rows: 21,459 from /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/outputs/parquets/yearly_si_nb06.parquet
Pathogens: 529; antibiotics/levels: 189
Saved cleaned GLASS rows: 214 to /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv
all screened AMR input pairs pathogens: 529
Notebook 06 yearly SI input pathogens: 529


,pathogen,source_rows,source_publications,source_pairs,modeled_rows,modeled_publications,modeled_pairs,in_source,in_modeled,status


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter baumannii,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter baumannii,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter baumannii,Doripenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter baumannii,Gentamicin,True,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter baumannii,Imipenem/cilastatin,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter baumannii,Meropenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter baumannii,Minocycline,True,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter baumannii,Tigecycline,True,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,Acinetobacter spp.,amikacin,Acinetobacter bereziniae,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
9,Acinetobacter spp.,Colistin,Acinetobacter bereziniae,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified


## Build Or Load Rank Data

In [5]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    source_path=YEARLY_SI_NB06_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    curve_fit_end_year=CURVE_FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    min_amr_pair_publication_count=MIN_AMR_PAIR_PUBLICATION_COUNT,
    min_num_years_with_valid_counts=MIN_NUM_YEARS_WITH_VALID_COUNTS,
    verbose=True,
)

num_ranked_pairs = len(rank_df)
rank_df = rank_df.loc[rank_df["status"].eq("ok")].copy()
num_successful_fits = len(rank_df)
rank_df = rank_df.loc[
    pd.to_numeric(rank_df["latest_available_predicted_SI"], errors="coerce") >= MIN_PLOT_SI
].copy()
print(f"Successfully fitted pairs: {num_successful_fits:,} / {num_ranked_pairs:,}")
print(f"Pairs with latest predicted SI >= {MIN_PLOT_SI:g}: {len(rank_df):,}")
display(rank_df)

Recomputing rank data because /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/outputs/parquets/yearly_si_ranked_future_nb06.parquet is older than /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/outputs/parquets/yearly_si_nb06.parquet
[1/5826] ranking Abiotrophia defectiva vs Clindamycin
[2/5826] ranking Abiotrophia defectiva vs Daptomycin
[3/5826] ranking Abiotrophia defectiva vs Erythromycin
[4/5826] ranking Abiotrophia defectiva vs Minocycline
[5/5826] ranking Abiotrophia defectiva vs Tetracycline
[6/5826] ranking Achromobacter mucicolens vs Amoxicillin
[7/5826] ranking Achromobacter mucicolens vs Ampicillin
[8/5826] ranking Achromobacter mucicolens vs Benzylpenicillin
[9/5826] ranking Achromobacter piechaudii vs Ceftriaxone
[10/5826] ranking Achromobacter xylosoxidans vs Amikacin
[11/5826] ranking Achromobacter xylosoxidans vs Amoxicillin/clavulanic-acid
[12/5826] ranking Achromobacter xylosoxidans vs Ampicillin
[13/5826]

,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,min_amr_pair_publication_count,...,tail_damping_years,validation_loess_frac,validation_loess_degree,validation_tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
5205,Escherichia coli,Clindamycin,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,536.0,251.0,285.0,285.0
5206,Yersinia enterocolitica,Amoxicillin/clavulanic-acid,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,286.0,486.0,-200.0,200.0
5207,Streptococcus pneumoniae,Meropenem,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,451.0,254.0,197.0,197.0
5208,Escherichia coli,Benzylpenicillin,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,513.0,327.0,186.0,186.0
5209,Bacillus cereus,Rifampicin,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,359.0,532.0,-173.0,173.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5821,Salmonella enterica,Ampicillin,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,18.0,18.0,0.0,0.0
5822,Klebsiella pneumoniae,Gentamicin,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,93.0,93.0,0.0,0.0
5823,Staphylococcus pseudintermedius,Chloramphenicol,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,402.0,402.0,0.0,0.0
5824,Stenotrophomonas maltophilia,Piperacillin/tazobactam,2025,2028,3,2000,2023,2025,2025,1.0,...,NaN,0.6,1,NaN,ok,None,456.0,456.0,0.0,0.0


## Interactive Latest Count Vs Future SI Shift

In [6]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_total_publications=MIN_TOTAL_PUBLICATIONS,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "interactive_latest_observed_count_vs_future_si_shift.pdf",
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")
if latest_count_shift_result["pdf_path"]:
    print(f"Saved PDF: {latest_count_shift_result['pdf_path']}")
elif latest_count_shift_result.get("pdf_error"):
    print(f"PDF export failed: {latest_count_shift_result['pdf_error']}")

Displayed rows: 244; GLASS matches: 51
Saved PDF: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/outputs/figures/interactive_latest_observed_count_vs_future_si_shift.pdf


## Display GLASS Matches In Plot

In [7]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Escherichia coli,Ciprofloxacin,2025.0,2028,113.0,1382.0,116.194960,0.965889,1.028844,0.062955,16.333888,antibiotic
1,Klebsiella pneumoniae,Colistin,2025.0,2028,67.0,659.0,77.847486,0.946036,0.995611,0.049575,6.230819,antibiotic
2,Escherichia coli,Sulfamethoxazole/trimethoprim,2025.0,2028,67.0,1054.0,88.070813,0.911683,0.961162,0.049479,14.080779,antibiotic
3,Escherichia coli,Colistin,2025.0,2028,60.0,629.0,59.536312,0.786090,0.763127,-0.022963,53.149942,antibiotic
4,Escherichia coli,Cefotaxime,2025.0,2028,59.0,697.0,82.822951,0.926159,1.015046,0.088886,24.187009,antibiotic
5,Escherichia coli,Ceftriaxone,2025.0,2028,58.0,444.0,78.624557,0.813977,0.978502,0.164525,19.672606,antibiotic
6,Salmonella enterica,Ampicillin,2025.0,2028,45.0,673.0,47.086513,0.755316,0.820353,0.065037,10.712829,antibiotic
7,Staphylococcus aureus,Oxacillin,2025.0,2028,44.0,898.0,45.435978,0.724637,0.742102,0.017466,4.833629,antibiotic
8,Pseudomonas aeruginosa,Meropenem,2025.0,2028,34.0,433.0,41.331731,0.744221,0.803030,0.058808,22.748788,antibiotic
9,Klebsiella pneumoniae,Meropenem,2025.0,2028,31.0,325.0,46.471955,0.720935,0.799434,0.078499,35.880566,antibiotic


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [8]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 193


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Vibrio parahaemolyticus,Cefazolin,2025.0,2028,4.0,22.0,9.221876,0.302105,0.648850,0.346745,73.620425
1,Escherichia coli,Clindamycin,2025.0,2028,5.0,28.0,10.570940,-0.240810,0.077009,0.317819,46.137652
2,Klebsiella pneumoniae,Amoxicillin/clavulanic-acid,2025.0,2028,16.0,69.0,32.019884,0.447082,0.757178,0.310096,15.594351
3,Escherichia coli,Cefixime,2025.0,2028,9.0,59.0,18.991200,0.301544,0.559240,0.257696,13.703878
4,Staphylococcus aureus,Amoxicillin,2025.0,2028,14.0,96.0,30.549025,0.402200,0.654316,0.252115,46.944239
...,...,...,...,...,...,...,...,...,...,...,...
188,Listeria monocytogenes,Ampicillin,2025.0,2028,5.0,53.0,2.156697,-0.026700,-0.107769,-0.081069,38.991856
189,Acinetobacter baumannii,Ceftazidime,2025.0,2028,8.0,181.0,5.771053,0.091612,0.007509,-0.084103,28.976451
190,Klebsiella pneumoniae,Polymyxin-B,2025.0,2028,6.0,91.0,6.456777,0.273654,0.189119,-0.084535,58.642180
191,Clostridioides difficile,Ciprofloxacin,2025.0,2028,4.0,92.0,2.134277,-0.093792,-0.212398,-0.118606,53.932711


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [9]:
latest_count_shift_df

,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,min_amr_pair_publication_count,...,latest_count_source,latest_smoothed_SI,future_minus_latest_smoothed_SI,count_error_display,marker_count_error,is_glass_pair,glass_group_column,appears_in_GLASS,label,display_label
0,Escherichia coli,Clindamycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,-0.240810,0.317819,46.14%,46.137652,False,antibiotic,no,E. coli<br>Clindamycin,E. coli<br>Clindamycin
1,Klebsiella pneumoniae,Streptomycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,-0.160128,0.155211,57.72%,57.722155,False,antibiotic,no,K. pneumoniae<br>Streptomycin,K. pneumoniae<br>Streptomycin
2,Salmonella enterica,Cefuroxime,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,-0.022653,0.205619,49.04%,49.040443,False,antibiotic,no,S. enterica<br>Cefuroxime,S. enterica<br>Cefuroxime
3,Klebsiella pneumoniae,Fosfomycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.110788,-0.119807,58.11%,58.109171,False,antibiotic,no,K. pneumoniae<br>Fosfomycin,K. pneumoniae<br>Fosfomycin
4,Klebsiella pneumoniae,Amoxicillin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,-0.063274,0.142639,61.32%,61.321841,False,antibiotic,no,K. pneumoniae<br>Amoxicillin,K. pneumoniae<br>Amoxicillin
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,Enterococcus faecium,Vancomycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,1.168652,0.052797,13.66%,13.657715,False,antibiotic,no,E. faecium<br>Vancomycin,E. faecium<br>Vancomycin
240,Escherichia coli,Ciprofloxacin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.965889,0.062955,16.33%,16.333888,True,antibiotic,yes,E. coli<br>Ciprofloxacin,E. coli<br>Ciprofloxacin
241,Escherichia coli,Cefotaxime,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.926159,0.088886,24.19%,24.187009,True,antibiotic,yes,E. coli<br>Cefotaxime,E. coli<br>Cefotaxime
242,Salmonella enterica,Ampicillin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.755316,0.065037,10.71%,10.712829,True,antibiotic,yes,S. enterica<br>Ampicillin,S. enterica<br>Ampicillin


In [10]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    predicted_future_si_threshold=0.0,
    predicted_future_delta_si_threshold=0.0,
    show_mean_count_pct_error=SHOW_MEAN_COUNT_PCT_ERROR,
    show_label_background=SHOW_LABEL_BACKGROUND,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    avoid_label_occlusion=True,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "future_si_vs_current_si_quadrant.pdf",
    quadrant_output_dir=FIGURE_OUTPUT_DIR / "future_si_quadrants",
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]
if future_si_shift_result["pdf_path"]:
    print(f"Saved PDF: {future_si_shift_result['pdf_path']}")
elif future_si_shift_result.get("pdf_error"):
    print(f"PDF export failed: {future_si_shift_result['pdf_error']}")
for quadrant_name, quadrant_result in future_si_shift_result["quadrants"].items():
    if quadrant_result["pdf_path"]:
        print(f"Saved {quadrant_name} quadrant: {quadrant_result['pdf_path']}")
    elif quadrant_result.get("pdf_error"):
        print(f"PDF export failed for {quadrant_name}: {quadrant_result['pdf_error']}")

Saved PDF: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/outputs/figures/future_si_vs_current_si_quadrant.pdf
Saved below_both quadrant: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/outputs/figures/future_si_quadrants/future_si_vs_si_shift_below_both.pdf
Saved below_future_si quadrant: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/outputs/figures/future_si_quadrants/future_si_vs_si_shift_below_future_si.pdf
Saved below_delta_si quadrant: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/outputs/figures/future_si_quadrants/future_si_vs_si_shift_below_delta_si.pdf
Saved above_both quadrant: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/outputs/figures/future_si_quadrants/future_si_vs_si_shift_above_both.pdf


In [11]:
latest_count_shift_df.to_csv(
    FIGURE_OUTPUT_DIR / "future_si_vs_current_si_quadrant.csv",
    index=False,
)

In [12]:
FIGURE_OUTPUT_DIR / "future_si_vs_current_si_quadrant.csv"

PosixPath('/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Antimicrobial-Resistance/outputs/figures/future_si_vs_current_si_quadrant.csv')

In [13]:
latest_count_shift_df[latest_count_shift_df['pathogen'] == 'Streptococcus pneumoniae']

,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,min_amr_pair_publication_count,...,latest_count_source,latest_smoothed_SI,future_minus_latest_smoothed_SI,count_error_display,marker_count_error,is_glass_pair,glass_group_column,appears_in_GLASS,label,display_label
37,Streptococcus pneumoniae,Sulfamethoxazole/trimethoprim,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.088758,-0.025532,1.24%,1.242403,True,antibiotic,yes,S. pneumoniae<br>Sulfamethoxazole/trimethoprim,S. pneumoniae<br>Sulfamethoxazole/trimethoprim
41,Streptococcus pneumoniae,Clindamycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.098718,-0.020968,70.71%,70.707371,False,antibiotic,no,S. pneumoniae<br>Clindamycin,S. pneumoniae<br>Clindamycin
101,Streptococcus pneumoniae,Erythromycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.580912,0.021660,36.21%,36.209964,True,antibiotic,yes,S. pneumoniae<br>Erythromycin,S. pneumoniae<br>Erythromycin
106,Streptococcus pneumoniae,Tetracycline,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.377655,0.047327,53.86%,53.858202,False,antibiotic,no,S. pneumoniae<br>Tetracycline,S. pneumoniae<br>Tetracycline
116,Streptococcus pneumoniae,Ceftriaxone,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.092420,0.014902,22.46%,22.461943,True,antibiotic,yes,S. pneumoniae<br>Ceftriaxone,S. pneumoniae<br>Ceftriaxone
126,Streptococcus pneumoniae,Cefuroxime,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.010113,0.051838,60.81%,60.807385,False,antibiotic,no,S. pneumoniae<br>Cefuroxime,S. pneumoniae<br>Cefuroxime
170,Streptococcus pneumoniae,Cefotaxime,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,-0.054301,0.012319,40.08%,40.077313,True,antibiotic,yes,S. pneumoniae<br>Cefotaxime,S. pneumoniae<br>Cefotaxime
182,Streptococcus pneumoniae,Azithromycin,2025,2028,3,2000,2023,2025,2025,1.0,...,observed,0.152166,0.062100,40.79%,40.793920,False,antibiotic,no,S. pneumoniae<br>Azithromycin,S. pneumoniae<br>Azithromycin


## 2025 PMIDs For Plotted Pairs

One row per PMID linked to each plotted bacteria-antibiotic pair in 2025. Pairs without a 2025 PMID remain in the table with a blank PMID.

In [14]:
CURRENT_YEAR = 2025

plotted_pairs_df = latest_count_shift_df[["pathogen", "antibiotic"]].drop_duplicates()
pmids_current_year_df = yearly_screened_model_df.loc[
    pd.to_numeric(yearly_screened_model_df["year"], errors="coerce").eq(CURRENT_YEAR),
    ["pathogen", "antibiotic", "pmids"],
].copy()

pair_pmids_2025_df = plotted_pairs_df.merge(
    pmids_current_year_df,
    on=["pathogen", "antibiotic"],
    how="left",
).explode("pmids", ignore_index=True)
pair_pmids_2025_df = (
    pair_pmids_2025_df.rename(columns={"pathogen": "bacteria", "pmids": "PMID"})
    .assign(year=CURRENT_YEAR)
    [["bacteria", "antibiotic", "year", "PMID"]]
    .sort_values(["bacteria", "antibiotic", "PMID"], na_position="last")
    .reset_index(drop=True)
)

from IPython.display import HTML

def _pmid_link(pmid):
    if pd.isna(pmid):
        return ""
    pmid = str(pmid)
    return f'<a href="https://pubmed.ncbi.nlm.nih.gov/{pmid}/" target="_blank">{pmid}</a>'

display(HTML(pair_pmids_2025_df.to_html(index=False, escape=False, formatters={"PMID": _pmid_link})))

bacteria,antibiotic,year,PMID
Acinetobacter baumannii,Amikacin,2025,39675013
Acinetobacter baumannii,Amikacin,2025,39704504
Acinetobacter baumannii,Amikacin,2025,40150047
Acinetobacter baumannii,Amikacin,2025,40160221
Acinetobacter baumannii,Amikacin,2025,40221655
Acinetobacter baumannii,Amikacin,2025,40330064
Acinetobacter baumannii,Amikacin,2025,40488459
Acinetobacter baumannii,Amikacin,2025,41146018
Acinetobacter baumannii,Amikacin,2025,41301637
Acinetobacter baumannii,Amikacin,2025,42226979
